# Feature Matrix EDA — Customer-level Features for TARGET

Builds the customer-level feature matrix (via `home_credit_risk.features.build_feature_matrix`) that aggregates every auxiliary Home Credit table to one row per `SK_ID_CURR` and joins it onto `application_train`, then explores it to decide **which features are useful** and **what needs imputing**.

Two lineages are assembled (see `docs/INDIVIDUAL_DATASET_EDA_SUMMARY.md`):
- **Previous-application hub** — ledgers (credit card / installments / POS) consolidated
  per `SK_ID_PREV`, joined onto `previous_application`, rolled up to `SK_ID_CURR`.
- **Bureau lineage** — `bureau_balance` → `SK_ID_BUREAU` → bridged via `bureau` → customer.

No imputation is applied here — unmatched left joins leave nulls, which this EDA analyses.

**Sections:** 1. Setup & build · 2. Coverage · 3. Missingness by block · 4. Correlation with TARGET · 5. Top-feature distributions · 6. Train/test parity

## 1. Setup and Build the Feature Matrix

In [ ]:
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

sys.path.insert(0, "/Users/bharathregula/code/home_credit_risk_deeplearn/src")
from home_credit_risk.eda_utils import (
    analyze_correlations,
    analyze_numeric_features,
    display_basic_info,
    show_missing_patterns,
)
from home_credit_risk.features import build_feature_matrix

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(80)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

In [ ]:
datasets = Path("/Users/bharathregula/code/datasets/home-credit-default-risk")

app_train = pl.read_csv(datasets / "application_train.csv", infer_schema_length=10000)
fm = build_feature_matrix(app_train, datasets)

app_cols = set(app_train.columns)
feature_cols = [c for c in fm.columns if c not in app_cols]
print(f"Feature matrix: {fm.shape}  (driver had {app_train.width} cols)")
print(f"Engineered feature columns: {len(feature_cols)}")
print(f"Row count preserved: {fm.height == app_train.height}")

In [ ]:
display_basic_info(fm, df_name="Feature_Matrix")

## 2. Coverage

`HAS_*` flags mark whether a customer has any history in each lineage; the `PREV_HAS_*_RATE` features give the fraction of a customer's prior loans that had each ledger. Low coverage → those feature blocks will be mostly null (see section 3).

In [ ]:
coverage = {
    "HAS_PREV (any previous application)": fm["HAS_PREV"].mean(),
    "HAS_BUREAU (any bureau record)": fm["HAS_BUREAU"].mean(),
    "PREV_HAS_CC_RATE (loans w/ credit card)": fm["PREV_HAS_CC_RATE"].mean(),
    "PREV_HAS_INST_RATE (loans w/ installments)": fm["PREV_HAS_INST_RATE"].mean(),
    "PREV_HAS_POS_RATE (loans w/ POS)": fm["PREV_HAS_POS_RATE"].mean(),
}
for k, v in coverage.items():
    print(f"{k:44} {v:.1%}")

## 3. Missingness by Feature Block (imputation input)

Mean null-rate per block is the key input for the imputation plan. A block that is null because the customer simply *has no such history* should be imputed with intent (counts → 0, keep the `HAS_*` flag) rather than a blind median.

In [ ]:
blocks = {
    "PREV_ (all)": "PREV_",
    "PREV_CC (credit card)": "PREV_CC",
    "PREV_INST (installments)": "PREV_INST",
    "PREV_POS (POS)": "PREV_POS",
    "BUREAU_": "BUREAU_",
}
print(f"{'block':26} {'cols':>5} {'mean null-rate':>15}")
for label, pref in blocks.items():
    cols = [c for c in feature_cols if c.startswith(pref)]
    if cols:
        rate = sum(fm[c].null_count() / fm.height for c in cols) / len(cols)
        print(f"{label:26} {len(cols):5d} {rate:14.1%}")

In [ ]:
# Columns with the most missing data (drives imputation priority)
missing = show_missing_patterns(fm.select(feature_cols), threshold=0.0)
missing.head(20)

## 4. Correlation with TARGET

Which features carry signal. Individual aggregated features are weak (this is a hard problem), but the credit-card usage and previous-application recency blocks stand out.

In [ ]:
# Overall (engineered + application features) — nulls median-filled inside the helper
correlations = analyze_correlations(fm, target_col="TARGET", top_n=15)

In [ ]:
# Ranking restricted to the ENGINEERED features
eng = fm.select(["TARGET"] + feature_cols).with_columns(
    [pl.col(c).fill_null(pl.col(c).median()) for c in feature_cols]
)
rows = []
for c in feature_cols:
    r = eng.select(pl.corr("TARGET", c)).item()
    # r is None for an all-null column and NaN for a constant (zero-variance)
    # column; skip both so they don't pollute the ranking.
    if r is not None and not math.isnan(r):
        rows.append({"feature": c, "corr": r})
eng_corr = (
    pl.DataFrame(rows)
    .with_columns(pl.col("corr").abs().alias("abs_corr"))
    .sort("abs_corr", descending=True)
)
print(
    f"Top 20 engineered features by |correlation| with TARGET (of {len(rows)} ranked):"
)
eng_corr.head(20)

### Categorical & windowed blocks

Iteration 2 added **one-hot rate** encodings of the `previous_application` / `bureau`
categoricals (`*_RATE`, `*_NUNIQUE`) and **recent time-window** aggregations (`*_L12M`,
`*_L365D`, `*_L730D`). Several of these rank among the strongest engineered signals —
notably the previous-application **refusal rate** and the recent **installment late-rate**.

In [ ]:
# Column counts by iteration-2 block type
counts = {"categorical (RATE/NUNIQUE)": 0, "windowed (L12M/L365D/L730D)": 0, "other": 0}
for c in feature_cols:
    if c.endswith("_RATE") or c.endswith("_NUNIQUE"):
        counts["categorical (RATE/NUNIQUE)"] += 1
    elif any(w in c for w in ("_L12M", "_L365D", "_L730D")):
        counts["windowed (L12M/L365D/L730D)"] += 1
    else:
        counts["other"] += 1
for k, v in counts.items():
    print(f"{k:32} {v}")

# Top categorical + windowed features by |correlation| with TARGET (reuse eng_corr)
new_block = eng_corr.filter(
    pl.col("feature").str.ends_with("_RATE")
    | pl.col("feature").str.ends_with("_NUNIQUE")
    | pl.col("feature").str.contains("_L12M")
    | pl.col("feature").str.contains("_L365D")
    | pl.col("feature").str.contains("_L730D")
)
print("\nTop iteration-2 features by |correlation| with TARGET:")
new_block.head(15)

## 5. Distributions of the Top Engineered Features

In [ ]:
top_feats = eng_corr.head(8)["feature"].to_list()
_ = analyze_numeric_features(fm.select(top_feats), show_summary=True)

## 6. Train / Test Parity

The same pipeline must produce the same columns for `application_test` (minus `TARGET`), so the model can score new customers.

In [ ]:
app_test = pl.read_csv(datasets / "application_test.csv", infer_schema_length=10000)
fm_test = build_feature_matrix(app_test, datasets)

train_cols = set(fm.columns) - {"TARGET"}
test_cols = set(fm_test.columns)
print(f"test matrix: {fm_test.shape}")
print(f"columns match (train minus TARGET == test): {train_cols == test_cols}")
print(f"only in train: {sorted(train_cols - test_cols)}")
print(f"only in test:  {sorted(test_cols - train_cols)}")

### Optional: persist for modeling

Uncomment to save the assembled matrices as parquet (keeps dtypes, fast reload).

In [ ]:
# fm.write_parquet(datasets.parent / 'features_train.parquet')
# fm_test.write_parquet(datasets.parent / 'features_test.parquet')